# Stochastic Calculus, Step by Step · Lesson 3: Probability basics

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. A share with three outcomes · 3. A continuous price: density and E[g(X)] · 4. Linearity and the variance formula · 5. Covariance and independence · 6. Exercises · 7. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 3,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. A share with three outcomes

In a year the share is worth 80, 100 or 125, with probabilities 0.25, 0.45 and 0.30. A random variable is just this: a number for each outcome, with a probability for each.

In [ ]:
vals = np.array([80.0, 100.0, 125.0]); probs = np.array([0.25, 0.45, 0.30])
mean = float((vals * probs).sum())
second = float((vals**2 * probs).sum())
call = float((np.maximum(vals - K, 0) * probs).sum())
out["discrete"] = {"vals": vals.tolist(), "probs": probs.tolist(), "mean": mean, "second": second, "var": second - mean**2,
                   "sd": math.sqrt(second - mean**2), "call": call, "p_above": float(probs[vals > K].sum()),
                   "call_at_mean": max(mean - K, 0.0)}
out["discrete"]["rows"] = [{"s": v, "p": q, "payoff": max(v - K, 0.0), "s_p": v * q, "payoff_p": max(v - K, 0.0) * q}
                           for v, q in zip(vals.tolist(), probs.tolist())]
out["discrete"]

## 3. A continuous price: density and E[g(X)]

A continuous price has a **density** $f$: probabilities are areas, $P(a < X < b) = \int_a^b f(x)\,dx$, and averages are weighted areas, $E[g(X)] = \int g(x)f(x)\,dx$. Take a triangular density on $[70, 140]$ peaking at 100. The call payoff $g(x) = \max(x - 105, 0)$ has

In [ ]:
a_, c_, b_ = 70.0, 100.0, 140.0
def tri_pdf(x):
    x = np.asarray(x, float)
    return np.where(x < c_, 2 * (x - a_) / ((b_ - a_) * (c_ - a_)), 2 * (b_ - x) / ((b_ - a_) * (b_ - c_))) * ((x >= a_) & (x <= b_))
xg = np.linspace(a_, b_, 70_001)
w = tri_pdf(xg)
integ = lambda y: float(((y[1:] + y[:-1]) / 2 * np.diff(xg)).sum())
tri = {"a": a_, "c": c_, "b": b_, "total": integ(w), "mean_exact": (a_ + b_ + c_) / 3,
       "mean_num": integ(xg * w), "var_exact": (a_**2 + b_**2 + c_**2 - a_ * b_ - a_ * c_ - b_ * c_) / 18,
       "p_above_exact": (b_ - K) ** 2 / ((b_ - a_) * (b_ - c_)), "p_above_num": integ(w * (xg > K)),
       "call_exact": (b_ - K) ** 3 / (3 * (b_ - a_) * (b_ - c_)), "call_num": integ(np.maximum(xg - K, 0) * w)}
tri["call_at_mean"] = max(tri["mean_exact"] - K, 0.0)
# the pieces of the mean integral, left and right of the peak, and the peak height
tri.update({"height": 2 / (b_ - a_), "area_left": (c_ - a_) / (b_ - a_), "left_slope_den": (c_ - a_) * (b_ - a_) / 2,
            "right_slope_den": (b_ - c_) * (b_ - a_) / 2,
            "mean_left": integ(xg * w * (xg <= c_)), "mean_right": integ(xg * w * (xg > c_))})
tri["sd_exact"] = math.sqrt(tri["var_exact"])
assert abs(tri["total"] - 1) < 1e-6 and abs(tri["call_num"] - tri["call_exact"]) < 1e-6
# Monte Carlo: inverse transform
rng = np.random.default_rng(seed)
u = rng.random(1_000_000)
split = (c_ - a_) / (b_ - a_)
X = np.where(u < split, a_ + np.sqrt(u * (b_ - a_) * (c_ - a_)), b_ - np.sqrt((1 - u) * (b_ - a_) * (b_ - c_)))
tri.update({"mc_mean": float(X.mean()), "mc_var": float(X.var()), "mc_call": float(np.maximum(X - K, 0).mean())})
out["tri"] = tri
out["tri_fig"] = {"x": xg[::500].round(3).tolist(), "pdf": tri_pdf(xg[::500]).round(6).tolist(),
                  "ticks": [{"x": v, "label": f"${v:g}"} for v in (70, 80, 90, 100, 110, 120, 130, 140)]}
pd.Series({k: v for k, v in tri.items()})

## 4. Linearity and the variance formula

$E[aX + bY] = aE[X] + bE[Y]$ always; $\mathrm{Var}(X) = E[X^2] - (E[X])^2$; $\mathrm{Var}(aX + b) = a^2\mathrm{Var}(X)$. A position of 2 shares and 50 in cash, on the triangular price:

In [ ]:
pos = 2 * X + 50
out["linear"] = {"mean": float(pos.mean()), "mean_rule": 2 * tri["mean_exact"] + 50,
                 "var": float(pos.var()), "var_rule": 4 * tri["var_exact"]}
out["linear"]

## 5. Covariance and independence

$\mathrm{Cov}(X, Y) = E[XY] - E[X]E[Y]$, and $\mathrm{Var}(X + Y) = \mathrm{Var}(X) + \mathrm{Var}(Y) + 2\,\mathrm{Cov}(X, Y)$. Independence gives zero covariance, but zero covariance does not give independence: with $U$ uniform on $[-1, 1]$ and $V = U^2$, the covariance is 0 and yet $V$ is a function of $U$.

In [ ]:
U = rng.uniform(-1, 1, 1_000_000)
V = U**2
Y = 0.5 * (X - tri["mean_exact"]) + rng.normal(0, 10, X.size)        # a second price that moves with X
cov_xy = float(np.cov(X, Y, ddof=0)[0, 1])
out["cov"] = {"cov_uv": float(np.cov(U, V, ddof=0)[0, 1]), "corr_uv": float(np.corrcoef(U, V)[0, 1]),
              "p_v_big_given_u_big": float((V[np.abs(U) > 0.5] > 0.25).mean()), "p_v_big": float((V > 0.25).mean()),
              "cov_xy": cov_xy, "cov_xy_exact": 0.5 * tri["var_exact"], "corr_xy": float(np.corrcoef(X, Y)[0, 1]),
              "sd_y": float(Y.std()),
              "var_sum": float((X + Y).var()), "var_sum_rule": float(X.var() + Y.var() + 2 * cov_xy)}
out["cov"]

## 6. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 1: Solutions*.

**Exercise 1.** For the three-outcome share, compute $E[S]$, $\mathrm{Var}(S)$, $P(S > 105)$ and $E[\max(S - 105, 0)]$.

**Exercise 2.** If $X$ is uniform on $[80, 130]$, show that $E[\max(X - K, 0)] = (130 - K)^2/100$ for $80 \le K \le 130$. Evaluate it at $K = 105$.

**Exercise 3.** Show that $\mathrm{Var}(2X + 50) = 4\,\mathrm{Var}(X)$ and $\mathrm{Cov}(X, 2X + 50) = 2\,\mathrm{Var}(X)$.

**Exercise 4.** Let $Z$ take the values $-1, 0, 1$ with probability $\tfrac13$ each, and $Y = Z^2$. Show that $\mathrm{Cov}(Z, Y) = 0$ but $Z$ and $Y$ are not independent.

**Exercise 5.** For the triangular density, find the distribution function $F(x) = P(X \le x)$ on both sides of the peak, check that $F(100) = 3/7$ and $F(140) = 1$, and use $F$ to compute $P(90 < X < 110)$.

**Exercise 6.** Two share prices $X$ and $Y$ each have standard deviation 20, and their correlation is $\rho$. Find the standard deviation of $X + Y$ for $\rho = 1$, $0.5$, $0$ and $-1$. What does this say about holding both?

**Exercise 7.** (Code) Draw 1,000,000 samples from the triangular density (inverse transform) and check $E[X]$, $\mathrm{Var}(X)$ and $E[\max(X - 105, 0)]$ against the exact values.

**Exercise 8.** (Code) Build two correlated prices and check $\mathrm{Var}(X - Y) = \mathrm{Var}(X) + \mathrm{Var}(Y) - 2\,\mathrm{Cov}(X, Y)$ by simulation.

## 7. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")